# 神經網路

神經網路是由很多個「神經元」接起來的。第一部先看一個神經元：它其實就是你學過的線性回歸和邏輯回歸。第二部換到烘咖啡：用兩層神經元判斷一爐咖啡烘得好不好。第三部把「不好」再分成太生和太過，答案從兩類變成三類，輸出層改用 softmax。

程式已經寫好，用來讓你看懂模型在做什麼。請 agent 改的格子，標題會寫「實驗」。下面這種格子只有你能寫，agent 不要填、不要改寫：

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>

</div>


## 給 agent

- 可以改標題含「實驗」的程式格，也可以依學生當下的問題加新的程式格。
- 不要改標題是「你的觀察」的格子，也不要替學生寫結論。
- 不要改 2.3、3.3 的學習率、epoch、複製份數，也不要拿掉 2.4 存好的權重，除非學生明確說要比較這一次訓練的結果。
- 「畫圖工具」那一格只負責畫圖和造資料，學生不用讀；不要把它的內容搬進各知識點的格子。
- 學生若只說「幫我做完」，先問他想把門檻調高還是調低，再改程式。

## 畫圖工具（不用看）

下一格只放畫圖和造資料的函式，還有要用的套件。先跑一次就好，不用讀懂。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

tf.get_logger().setLevel("ERROR")

GOOD = dict(marker="x", c="red")
BAD = dict(marker="o", facecolors="none", edgecolors="#0D5BDC")

def plot_line_fit(X, Y, w, b):
    x = np.linspace(0, X.max() * 1.1, 50)
    plt.scatter(X, Y, s=80, **GOOD, label="cities")
    plt.plot(x, w[0, 0] * x + b[0], label=f"w={w[0, 0]}, b={b[0]}")
    plt.xlabel("Population (10,000s)")
    plt.ylabel("Profit ($10,000s)")
    plt.legend()
    plt.show()

def plot_sigmoid_fit(X, Y, w, b):
    x = np.linspace(X.min() - 10, X.max() + 10, 200)
    pos = Y.reshape(-1) == 1
    plt.scatter(X[pos], Y[pos], s=80, **GOOD, label="admitted")
    plt.scatter(X[~pos], Y[~pos], s=80, **BAD, label="not admitted")
    plt.plot(x, 1 / (1 + np.exp(-(w[0, 0] * x + b[0]))), label=f"w={w[0, 0]}, b={b[0]}")
    plt.xlabel("Exam score")
    plt.ylabel("chance of admission")
    plt.legend()
    plt.show()

def load_coffee_data():
    rng = np.random.default_rng(2)
    X = rng.random(400).reshape(-1, 2)
    X[:, 1] = X[:, 1] * 4 + 11.5
    X[:, 0] = X[:, 0] * (285 - 150) + 150
    Y = np.zeros(len(X))
    for i, (temperature, duration) in enumerate(X):
        limit = -3 / (260 - 175) * temperature + 21
        good = 175 < temperature < 260 and 12 < duration < 15 and duration <= limit
        Y[i] = 1 if good else 0
    return X, Y.reshape(-1, 1)

def _good_region(ax):
    temperature = np.linspace(175, 260, 50)
    ax.plot(temperature, (-3 / 85) * temperature + 21, color="#7030A0", linewidth=1)
    ax.axhline(12, color="#7030A0", linewidth=1)
    ax.axvline(175, color="#7030A0", linewidth=1)
    ax.set_xlabel("Temperature (Celsius)")

def _coffee_axes(ax, X, Y, size=40):
    good = Y.reshape(-1) == 1
    ax.scatter(X[good, 0], X[good, 1], s=size, **GOOD, label="good")
    ax.scatter(X[~good, 0], X[~good, 1], s=size, **BAD, label="bad")
    _good_region(ax)

def _coffee_grid(n=50):
    tt, dd = np.meshgrid(np.linspace(150, 285, n), np.linspace(11.5, 15.5, n))
    return tt, dd, np.column_stack([tt.ravel(), dd.ravel()])

def plot_coffee(X, Y):
    fig, ax = plt.subplots()
    _coffee_axes(ax, X, Y, size=70)
    ax.set_ylabel("Duration (minutes)")
    ax.legend(loc="upper right")
    plt.show()

def plot_loss(history):
    plt.plot(history.history["loss"], marker="o")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.show()

def plot_units(X, Y, norm_l, W1, b1):
    tt, dd, raw = _coffee_grid(40)
    xn = np.asarray(norm_l(raw))
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for unit, ax in enumerate(axes):
        out = 1 / (1 + np.exp(-(xn @ W1[:, unit] + b1[unit])))
        shade = ax.pcolormesh(tt, dd, out.reshape(tt.shape), cmap="Blues", vmin=0, vmax=1, shading="nearest")
        _coffee_axes(ax, X, Y, size=30)
        ax.set_title(f"layer 1, unit {unit}")
    axes[0].set_ylabel("Duration (minutes)")
    fig.colorbar(shade, ax=axes.ravel().tolist())
    plt.show()

def plot_threshold(model, norm_l, X, Y, threshold):
    tt, dd, raw = _coffee_grid(50)
    prob = model.predict(norm_l(raw), verbose=0).reshape(tt.shape)
    fig, ax = plt.subplots()
    ax.contourf(tt, dd, prob >= threshold, levels=[-0.1, 0.5, 1.1], colors=["#d6e6ff", "#ffd6d6"], alpha=0.8)
    _coffee_axes(ax, X, Y)
    ax.set_title(f"threshold {threshold}")
    ax.set_ylabel("Duration (minutes)")
    ax.legend(loc="upper right")
    plt.show()

LEVEL_STYLE = [
    dict(marker="o", facecolors="none", edgecolors="#2a9d8f", label="under"),
    dict(marker="x", c="red", label="good"),
    dict(marker="s", facecolors="none", edgecolors="#6b3e26", label="over"),
]

def _level_axes(ax, X, Y3, size=40):
    for level, style in enumerate(LEVEL_STYLE):
        mask = Y3 == level
        ax.scatter(X[mask, 0], X[mask, 1], s=size, **style)
    _good_region(ax)
    ax.set_ylabel("Duration (minutes)")
    ax.legend(loc="upper right")

def plot_levels(X, Y3):
    fig, ax = plt.subplots()
    _level_axes(ax, X, Y3, size=60)
    plt.show()

def plot_level_regions(model, norm_l, X, Y3, title):
    tt, dd, raw = _coffee_grid(80)
    level = model.predict(norm_l(raw), verbose=0).argmax(axis=1).reshape(tt.shape)
    fig, ax = plt.subplots()
    ax.contourf(tt, dd, level, levels=[-0.5, 0.5, 1.5, 2.5], colors=["#d9f2ee", "#ffd6d6", "#eadfd6"], alpha=0.9)
    _level_axes(ax, X, Y3)
    ax.set_title(title)
    plt.show()

print("工具準備好了")

## 第一部：一個神經元

神經網路裡最小的零件是一個神經元（也叫單元）。這一部用 Keras 的 `Dense` 做出一個神經元，放進我們指定的 `w`、`b`，再跟你學過的公式對答案。

這一部的順序：

- **1.1** 沒有 activation 的神經元 = 線性回歸
- **1.2** 加上 sigmoid 的神經元 = 邏輯回歸

### 1.1 沒有 activation 的神經元 = 線性回歸

回到線性回歸那本的餐廳。這裡只拿兩個城市：人口 5 萬人的店每月賺 2 萬美元，人口 10 萬人的店每月賺 8 萬美元。

一個沒有 activation 的神經元，算的就是線性回歸：

$$
f_{w,b}(x) = wx + b
$$

![一個沒有 activation 的神經元：人口乘上 w 再加 b，得到獲利](images/neuron_linear.png)

`Dense(units=1)` 就是一個神經元。剛建好時 `w`、`b` 是隨機的，下一格把它們換成 `w = 1.2`、`b = -4`，再比 Keras 的輸出和公式算的。

In [ ]:
X_city = np.array([[5.0], [10.0]])   # 人口（萬人）
Y_city = np.array([[2.0], [8.0]])    # 每月獲利（萬美元）

linear_layer = Dense(units=1, activation="linear")
linear_layer(X_city)                 # 先算一次，Keras 才會建出 w、b
print("剛建好的 w、b（隨機）", linear_layer.get_weights())

w = np.array([[1.2]])
b = np.array([-4.0])
linear_layer.set_weights([w, b])
print("Keras 的輸出 ", np.asarray(linear_layer(X_city)).reshape(-1))
print("公式 wx + b  ", (X_city @ w + b).reshape(-1))

plot_line_fit(X_city, Y_city, w, b)

### 1.2 加上 sigmoid 的神經元 = 邏輯回歸

回到邏輯回歸那本的招生處，這次只看一次考試。6 位申請人的分數是 35 到 85，60 分以上的都錄取了。

神經元加上 sigmoid，算的就是邏輯回歸：

$$
f_{w,b}(x) = g(wx + b) \qquad g(z) = \frac{1}{1+e^{-z}}
$$

![加上 sigmoid 的神經元：分數先算出 z，再經過 sigmoid 變成錄取機會](images/neuron_sigmoid.png)

下一格放進 `w = 0.2`、`b = -12`。`wx + b = 0` 的地方是 `x = 60`，所以 60 分時機會剛好是 0.5。

In [ ]:
X_exam = np.array([[35.0], [45.0], [55.0], [65.0], [75.0], [85.0]])   # 考試分數
Y_exam = np.array([[0], [0], [0], [1], [1], [1]])                     # 1 是錄取

logistic_layer = Dense(units=1, activation="sigmoid")
logistic_layer(X_exam)

w = np.array([[0.2]])
b = np.array([-12.0])
logistic_layer.set_weights([w, b])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("Keras 的輸出     ", np.asarray(logistic_layer(X_exam)).reshape(-1).round(3))
print("公式 g(wx + b)   ", sigmoid(X_exam @ w + b).reshape(-1).round(3))

plot_sigmoid_fit(X_exam, Y_exam, w, b)

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">1.1 和 1.2 的神經元，程式只差在哪裡：</li>
<li style="color:#1a1a1a;">如果把 1.2 的 <code style="color:#1a1a1a;">b</code> 改成 <code style="color:#1a1a1a;">-14</code>，機會剛好 0.5 的分數會往哪邊移：</li>
</ol>

</div>

## 第二部：這一爐烘得好不好

![烘焙師看著溫度和時間，判斷這一爐咖啡烘得好不好](images/roasting.jpg)

你在烘咖啡。每一爐記下溫度（°C）和時間（分鐘），結果是好烘（`y = 1`）或不好（`y = 0`）。好烘大致落在 175–260°C、12–15 分鐘這一塊，而且時間不能比溫度允許的上限更長。

一個神經元只能畫一條直線，這塊區域要用好幾個神經元接起來。網路有兩層：第一層 3 個 sigmoid 單元，第二層 1 個 sigmoid 單元。

這一部最後要做出的，就是下面這條流程：把一爐的溫度和時間先正規化，經過第一層 3 個單元，再由第二層算出好烘機會，最後跟門檻比。圖上的數字用的是 2.4 存好的那組權重。

![一爐咖啡從溫度、時間到預測結果的流程](images/flow.png)

這一部的順序：

- **2.1** 資料：好烘落在哪一塊
- **2.2** 正規化：把溫度和時間拉到差不多大
- **2.3** 搭兩層並訓練
- **2.4** 換上存好的權重，自己算一次
- **2.5** 每個單元管哪一塊
- **2.6** 實驗：改門檻

### 2.1 資料：好烘落在哪一塊

沒有資料檔。下一格用固定種子造 200 筆，所以每次圖一樣。紅色叉是好烘，藍色圈是不好。紫線標出造資料時用的那塊好烘區域，不是模型畫的。

In [ ]:
X, Y = load_coffee_data()
print("形狀", X.shape, Y.shape)
print("前 3 爐 [溫度, 時間]", X[:3].round(1).tolist(), "結果", Y[:3].reshape(-1))
print("好烘筆數", int(Y.sum()))

plot_coffee(X, Y)

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">好烘是不是集中在某一塊？</li>
<li style="color:#1a1a1a;">溫度太低或時間太長時，看起來怎樣？</li>
</ol>

</div>


### 2.2 正規化：把溫度和時間拉到差不多大

溫度是一百多度，時間是十幾分鐘。尺度差這麼多時，同一步長會被溫度拉著走。每個特徵減掉自己的平均，再除以自己的標準差：

$$
x_{\text{norm}} = \frac{x - \mu}{\sigma}
$$

Keras 的 `Normalization` 做的就是這件事。`adapt` 從這 200 筆算出平均和標準差，之後預測新的一爐，也要用同一組。

In [ ]:
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)
Xn = np.asarray(norm_l(X))

print("平均   ", np.asarray(norm_l.mean).reshape(-1).round(2))
print("標準差 ", np.sqrt(np.asarray(norm_l.variance)).reshape(-1).round(2))
print("正規化前 溫度", X[:, 0].min().round(1), "~", X[:, 0].max().round(1), " 時間", X[:, 1].min().round(1), "~", X[:, 1].max().round(1))
print("正規化後 溫度", Xn[:, 0].min().round(2), "~", Xn[:, 0].max().round(2), " 時間", Xn[:, 1].min().round(2), "~", Xn[:, 1].max().round(2))

### 2.3 搭兩層並訓練

![烘咖啡的網路：溫度、時間接到第一層 3 個單元，再接到第二層 1 個單元](images/network.png)

`Sequential` 把兩層 `Dense` 接起來。`summary` 會列出每層有幾個參數：第一層 `2×3 + 3 = 9`，第二層 `3×1 + 1 = 4`。

損失是二元交叉熵，就是邏輯回歸用的那個 loss；優化器是 Adam，梯度下降的一種。訓練把這 200 筆複製 1000 份，讓每個 epoch 多看幾次同樣的例子。學習率 `0.01`，跑 10 個 epoch，這些數字不在這一格改。

In [ ]:
tf.random.set_seed(1234)
model = Sequential(
    [
        tf.keras.Input(shape=(2,)),
        Dense(3, activation="sigmoid", name="layer1"),
        Dense(1, activation="sigmoid", name="layer2"),
    ]
)
model.summary()

Xt = np.tile(Xn, (1000, 1))
Yt = np.tile(Y, (1000, 1))
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
)
history = model.fit(Xt, Yt, epochs=10, verbose=0)
print("每個 epoch 的損失", [round(float(v), 3) for v in history.history["loss"]])

plot_loss(history)

### 2.4 換上存好的權重，自己算一次

每次訓練出來的權重都不太一樣：三個單元沒有固定分工，這一次 0 號學到「太冷」，下一次可能改學「時間太短」。為了讓後面的圖每次都一樣，這裡換上一組存好的權重。

接著用 numpy 自己算一爐 `200°C、13.9 分鐘`：每一層都是 1.2 那個神經元，只是一次算好幾個。

$$
\vec{a}^{[1]} = g\left(\vec{x}\,W^{[1]} + \vec{b}^{[1]}\right) \qquad a^{[2]} = g\left(\vec{a}^{[1]}\,W^{[2]} + b^{[2]}\right)
$$

`W1` 是 2×3：2 個輸入、3 個單元，每一直行是一個單元的 `w`。

In [ ]:
W1 = np.array(
    [
        [-8.94, 0.29, 12.89],
        [-0.17, -7.34, 10.79],
    ]
)
b1 = np.array([-9.87, -9.28, 1.01])
W2 = np.array([[-31.38], [-27.86], [-32.79]])
b2 = np.array([15.54])
model.get_layer("layer1").set_weights([W1, b1])
model.get_layer("layer2").set_weights([W2, b2])

x = np.array([[200, 13.9]])
xn = np.asarray(norm_l(x))
a1 = sigmoid(xn @ W1 + b1)
a2 = sigmoid(a1 @ W2 + b2)
print("正規化後       ", xn.reshape(-1).round(2))
print("第一層 3 個單元", a1.reshape(-1).round(3))
print("第二層 好烘機會", a2.reshape(-1).round(3))
print("Keras 算的     ", model.predict(xn, verbose=0).reshape(-1).round(3))

### 2.5 每個單元管哪一塊

每張圖畫第一層一個單元在整片溫度、時間上的輸出，越藍越接近 1。

In [ ]:
plot_units(X, Y, norm_l, W1, b1)

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">三個單元裡，你看得出哪一個在管溫度太低、哪一個在管時間嗎：</li>
<li style="color:#1a1a1a;">第二層的 <code style="color:#1a1a1a;">W2</code> 三個數都是負的。單元輸出越大，好烘機會會變大還變小：</li>
</ol>

</div>

### 2.6 實驗：改門檻

下一格只改門檻。模型輸出的是「好烘」的機會，門檻放低，更多爐被判成好烘；放高則相反。兩筆試烘是 `200°C、13.9 分鐘`（落在好烘區裡）和 `200°C、17 分鐘`（時間太長）。

In [ ]:
# 實驗：改 threshold，看判成好烘的範圍變大還變小
threshold = 0.5

X_try = np.array([[200, 13.9], [200, 17]])
prob = model.predict(norm_l(X_try), verbose=0).reshape(-1)
for row, p in zip(X_try, prob):
    print(f"{row[0]:.0f}°C、{row[1]} 分鐘 → 機會 {p:.3f} →", "好烘" if p >= threshold else "不好")

plot_threshold(model, norm_l, X, Y, threshold)

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">門檻從 0.5 調高之後，<code style="color:#1a1a1a;">17 分鐘</code> 那一筆還是不好。<code style="color:#1a1a1a;">13.9 分鐘</code> 那一筆呢：</li>
<li style="color:#1a1a1a;">圖上判成好烘的顏色區域變大還變小。它有沒有蓋過紫線外面：</li>
<li style="color:#1a1a1a;">若你是烘焙師，你會把門檻放高還是放低：</li>
</ol>

</div>

## 第三部：不好是哪一種不好

烘焙師想知道的不只是「好不好」。烘不好的那一爐，是太生（溫度太低或時間太短），還是太過（溫度太高或時間太長）？這兩種的補救方法不一樣。

答案從兩類變成三類：太生 `0`、剛好 `1`、太過 `2`。第二部的輸出層只有一個 sigmoid，給的是「好烘」的機會。這裡要的是三個機會，而且三個加起來要是 1，這就是 softmax。

這一部的順序：

- **3.1** softmax：把幾個分數變成加起來是 1 的機會
- **3.2** 資料：不好分成太生和太過
- **3.3** 輸出層換成 3 個單元加 softmax
- **3.4** 實驗：拿掉隱藏層

### 3.1 softmax：把幾個分數變成加起來是 1 的機會

輸出層有幾類就放幾個單元，每個單元先算出一個分數 `z`。softmax 把每個分數取 `e` 的次方，再除以全部的總和：

$$
a_j = \frac{e^{z_j}}{\sum_{k} e^{z_k}}
$$

分數越大，機會越大；全部的機會加起來剛好是 1。下一格用 numpy 算一次，跟 TensorFlow 的 `tf.nn.softmax` 對答案。最後再看只有兩類時，softmax 和 sigmoid 的關係。

In [ ]:
def softmax(z):
    e = np.exp(z - np.max(z))   # 先減掉最大值，只是避免 exp 太大，結果不變
    return e / e.sum()

z = np.array([1.0, 2.0, 3.0])
a = softmax(z)
print("分數 z            ", z)
print("softmax 自己算    ", a.round(3), " 加起來", a.sum().round(3))
print("tf.nn.softmax     ", tf.nn.softmax(z).numpy().round(3))

z_two = np.array([0.0, 1.5])
print("只有兩類：softmax([0, 1.5]) 的第二個", softmax(z_two)[1].round(3), " sigmoid(1.5) =", sigmoid(1.5).round(3))

### 3.2 資料：不好分成太生和太過

還是第二部那 200 爐，只是重新標：好烘的照舊是 `1`；不好的裡面，溫度在 175°C 以下或時間在 12 分鐘以下的是太生 `0`，其他是太過 `2`。

In [ ]:
LEVEL_NAMES = ["太生", "剛好", "太過"]

under = (X[:, 0] <= 175) | (X[:, 1] <= 12)
Y3 = np.where(Y.reshape(-1) == 1, 1, np.where(under, 0, 2))
for level, name in enumerate(LEVEL_NAMES):
    print(f"{level} {name}：{int(np.sum(Y3 == level))} 爐")

plot_levels(X, Y3)

### 3.3 輸出層換成 3 個單元加 softmax

跟第二部比，只改兩個地方：

- 輸出層從 `Dense(1, activation="sigmoid")` 換成 `Dense(3, activation="softmax")`，一類一個單元。
- 損失從二元交叉熵換成 `SparseCategoricalCrossentropy`，答案直接給 `0`、`1`、`2` 就好。

正規化、隱藏層、Adam、學習率、複製份數、epoch 都跟 2.3 一樣。預測時，三個機會裡最大的那一類就是答案，不用門檻。下一格把訓練寫成 `train_levels`，3.4 會再用一次。

In [ ]:
def train_levels(hidden_units):
    tf.random.set_seed(1234)
    layers = [tf.keras.Input(shape=(2,))]
    if hidden_units > 0:
        layers.append(Dense(hidden_units, activation="sigmoid"))
    layers.append(Dense(3, activation="softmax"))
    model = Sequential(layers)
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        metrics=["accuracy"],
    )
    model.fit(np.tile(Xn, (1000, 1)), np.tile(Y3, 1000), epochs=10, verbose=0)
    loss, accuracy = model.evaluate(Xn, Y3, verbose=0)
    print(f"隱藏層 {hidden_units} 個單元：損失 {loss:.3f}，正確率 {accuracy * 100:.1f}%")
    return model

level_model = train_levels(3)

X_try3 = np.array([[160, 13.0], [200, 13.9], [200, 17]])
prob3 = level_model.predict(norm_l(X_try3), verbose=0)
for row, p in zip(X_try3, prob3):
    print(f"{row[0]:.0f}°C、{row[1]} 分鐘 → 太生 {p[0]:.3f}、剛好 {p[1]:.3f}、太過 {p[2]:.3f} → {LEVEL_NAMES[p.argmax()]}")

plot_level_regions(level_model, norm_l, X, Y3, "hidden layer: 3 units")

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">每一爐的三個機會加起來是多少：</li>
<li style="color:#1a1a1a;"><code style="color:#1a1a1a;">160°C、13 分鐘</code> 那一爐，模型判成哪一類？三個機會裡最大的是哪一個：</li>
<li style="color:#1a1a1a;">第二部用門檻判斷好不好，這裡沒有門檻。它是怎麼決定答案的：</li>
</ol>

</div>

### 3.4 實驗：拿掉隱藏層

`hidden_units = 0` 表示沒有隱藏層，只剩 softmax 輸出層。這時每一類的分數都只是溫度和時間的加權相加，跟一個神經元一樣只能畫直線。

下一格只改 `hidden_units`，看三區的分界和正確率怎麼變。

In [ ]:
# 實驗：改 hidden_units，0 表示拿掉隱藏層
hidden_units = 0

trial_model = train_levels(hidden_units)
plot_level_regions(trial_model, norm_l, X, Y3, f"hidden layer: {hidden_units} units")

<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">拿掉隱藏層之後，三區的分界變成什麼形狀：</li>
<li style="color:#1a1a1a;">正確率比 3.3 差多少。你覺得為什麼：</li>
<li style="color:#1a1a1a;">把 <code style="color:#1a1a1a;">hidden_units</code> 改成 <code style="color:#1a1a1a;">1</code>、再改成 <code style="color:#1a1a1a;">6</code> 各跑一次。隱藏單元的數量怎麼影響分界和正確率：</li>
</ol>

</div>

實驗本到這裡。接下來用瀏覽器打開同資料夾的 `委託.html`，在 `體能等級.ipynb` 裡做體能等級估計。